# Track 1 · Step 8b: top-k accuracy and ECE on test (report only, second opening of test)

AI251P Capstone, Phase 1. Owner: Abdulmohsen Alharthi. Decided 25 Oct: the rubric asks for top-k accuracy, and `2026-10-23_TestPredictions_v1.csv.gz` stores labels, not probabilities.

Rules of this notebook:
1. Uses **only** the 12 checkpoints listed in the first `TestLog.csv` entry, by exact file name.
2. Inference only (same preprocessing code as Step 8). Nothing is trained, tuned or selected; the logit bias is the one stored in each checkpoint (fitted on val).
3. Before any metric: raw and adjusted argmax must equal the `TestPredictions_v1` columns for every crop, model and seed. The only allowed difference is a near-tie: fp16 autocast (as in Step 8) can flip the argmax when the top-1 and top-2 logits of the new run differ by less than `TIE_GAP` (1e-2). These cases are counted and printed per model, seed and kind; any other difference stops the notebook before metrics or files.
4. top-1, top-2, top-3 accuracy and ECE (15 bins), raw and adjusted, full test and test without SeaDronesSee `rgb`, mean ± std over seeds.
5. Writes `2026-10-25_TestProbs_v1.csv.gz` and `2026-10-25_TestTopK_v1.csv` (never overwrites) and appends opening 2 to `TestLog.csv` with its reason.

**Run all once.** Everything that can fail without looking at test (GPU check, the 12 checkpoints built and loaded, test crops copied, loaded and length-checked, output names free) runs first. The TestLog line is written right before the first forward pass on test, because every look at test counts; a second run stops because that line exists.

Runtime: **GPU T4** (required: same hardware and fp16 autocast as Step 8).

## 0. Settings, mount Drive

In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'
SEEDS = [42, 43, 44]
EVAL = {'full FT': 'cls-full-64-s{seed}-lr0.002',      # as in step8_test_eval_2026-10-23_DONOTRERUN.ipynb
        'full FT lr0.001': 'cls-full-64-s{seed}',
        'scratch': 'cls-scratch-64-s{seed}',
        'frozen': 'cls-frozen-64-s{seed}'}
PRED_FILE = '2026-10-23_TestPredictions_v1.csv.gz'
STAMP = '2026-10-25'
REASON = 'report-only: top-k and ECE, same checkpoints, no tuning'
EXCLUDE = ('SeaDronesSee', 'rgb')
TOPK = (1, 2, 3)
TIE_GAP = 1e-2                   # argmax may differ from TestPredictions_v1 only if top1 - top2 logit < TIE_GAP (fp16 near-tie)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%writefile /content/aerial_crops.py
"""Aerial object crops for the AI251P 8-class classifier (Track 1).

Crops are stored by Step 4 as one uint8 array per split, shape (N, S, S, 3), plus an index CSV whose
`row` column points into the array. Labels follow CLASSES below.
"""
import numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision.transforms import v2 as T

CLASSES = ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy', 'car', 'truck', 'bus']
CLASS_TO_ID = {c: i for i, c in enumerate(CLASSES)}
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)   # ImageNet, used for scratch and pretrained alike


class Rot90:
    """Random multiple of 90 degrees: aerial views have no fixed 'up'."""
    def __call__(self, x):
        return torch.rot90(x, int(torch.randint(4, (1,))), dims=(-2, -1))


def make_transforms(img_size, train):
    if train:
        return T.Compose([T.ToImage(), T.RandomResizedCrop(img_size, scale=(0.75, 1.0), ratio=(0.9, 1.1), antialias=True),
                          T.RandomHorizontalFlip(), Rot90(),
                          T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
                          T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
    return T.Compose([T.ToImage(), T.Resize(img_size, antialias=True),
                      T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])


class CropDataset(Dataset):
    def __init__(self, array_path, index, transform=None):
        self.x = np.load(array_path, mmap_mode='r')
        self.index = index.sort_values('row').reset_index(drop=True)
        self.y = self.index.category.map(CLASS_TO_ID).to_numpy()
        assert len(self.index) == len(self.x) and not np.isnan(self.y.astype(float)).any()
        self.transform = transform

    def __len__(self): return len(self.index)

    def __getitem__(self, i):
        img = np.array(self.x[i])          # copy out of the read-only memmap
        if self.transform is not None: img = self.transform(img)
        return img, int(self.y[i])


def class_weights(labels, power=0.5):
    """w_c proportional to 1 / n_c**power, mean weight over samples = 1. power 0 = none, 1 = full balance."""
    n = np.bincount(labels, minlength=len(CLASSES)).astype(float)
    w = np.where(n > 0, 1.0 / np.maximum(n, 1) ** power, 0.0)
    return w / (w[labels].mean())


def seed_worker(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s); import random; random.seed(s)


def make_loader(ds, batch_size, train, seed=42, balance_power=0.5, num_workers=2):
    g = torch.Generator().manual_seed(seed)
    sampler = None
    if train and balance_power > 0:
        w = class_weights(ds.y, balance_power)[ds.y]
        sampler = WeightedRandomSampler(torch.as_tensor(w, dtype=torch.double), num_samples=len(ds), replacement=True, generator=g)
    return DataLoader(ds, batch_size=batch_size, shuffle=train and sampler is None, sampler=sampler,
                      num_workers=num_workers, worker_init_fn=seed_worker, generator=g,
                      pin_memory=torch.cuda.is_available(), drop_last=train, persistent_workers=num_workers > 0)

## 1. Checkpoints from the TestLog, crops, guards

In [ ]:
import os, re, sys, json, time, shutil, importlib
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, torchvision
sys.path.insert(0, '/content'); import aerial_crops as ac; importlib.reload(ac)
pd.set_option('display.width', 200)
ROOT = Path(DRIVE_PROJECT); OUT_DIR = ROOT / '02_Classification'; CKPT_DIR = OUT_DIR / 'Checkpoints'
CROP_DIR = ROOT / '01_Data/Processed/Crops'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda', 'no GPU: Runtime > Change runtime type > T4 GPU (Step 8 ran on a T4 with fp16 autocast)'
GPU = torch.cuda.get_device_name(0); assert 'T4' in GPU, f'GPU is {GPU}, Step 8 used a T4'
print('GPU:', GPU)
K = len(ac.CLASSES)
PROBS_OUT, TOPK_OUT = OUT_DIR / f'{STAMP}_TestProbs_v1.csv.gz', OUT_DIR / f'{STAMP}_TestTopK_v1.csv'
for f in (PROBS_OUT, TOPK_OUT): assert not f.exists(), f'{f.name} exists already: nothing is overwritten'

log_path = OUT_DIR / 'TestLog.csv'
LOG = pd.read_csv(log_path, keep_default_na=False)
print(f'TestLog: {len(LOG)} entr(ies)'); print(LOG.to_string(index=False))
assert 'reason' not in LOG.columns or REASON not in set(LOG['reason']), 'this report-only opening is already logged: not run twice'
assert len(LOG) == 1, 'expected exactly one earlier opening of test (23 Oct)'
NAMES = LOG.checkpoints.iloc[0].split(';')
assert len(NAMES) == len(EVAL) * len(SEEDS), f'{len(NAMES)} checkpoints in the TestLog, expected {len(EVAL) * len(SEEDS)}'

CKPTS = {}                       # (model, seed) -> path, matched by exact run name
for n in NAMES:
    run = re.fullmatch(r'\d{4}-\d{2}-\d{2}_(.+)_v\d+\.pt', n)[1]
    hit = [(m, s) for m, pat in EVAL.items() for s in SEEDS if pat.format(seed=s) == run]
    assert len(hit) == 1, f'{n}: matches {hit}'
    assert hit[0] not in CKPTS, f'two checkpoints for {hit[0]}'
    CKPTS[hit[0]] = CKPT_DIR / n; assert CKPTS[hit[0]].exists(), n
print(pd.Series({f'{m} s{s}': p.name for (m, s), p in CKPTS.items()}).to_string())

CK = {k: torch.load(p, map_location='cpu', weights_only=False) for k, p in CKPTS.items()}
used = {json.dumps(c['crops_used'], sort_keys=True) for c in CK.values()}
assert len(used) == 1, f'checkpoints trained on different crop files: {used}'
CROPS_USED = json.loads(used.pop())
for k, c in CK.items(): assert c['classes'] == ac.CLASSES and 'logit_bias' in c and c['seed'] == k[1], k

PRED = pd.read_csv(OUT_DIR / PRED_FILE, keep_default_na=False)
index = pd.read_csv(CROP_DIR / CROPS_USED['index'], keep_default_na=False)
TEST = index[index.split == 'test'].sort_values('row').reset_index(drop=True)
assert TEST.crop_id.tolist() == PRED.crop_id.tolist(), 'test crop order differs from TestPredictions_v1'
for k in CKPTS:
    for kind in ('raw', 'adjusted'): assert f'{k[0].replace(" ", "")}_s{k[1]}_{kind}' in PRED.columns, (k, kind)
YT = TEST.category.map(ac.CLASS_TO_ID).to_numpy()

## 2. Models and test crops, all loaded before the TestLog line (no forward pass yet)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_classes=8, width=32, dropout=0.3):
        super().__init__()
        layers, c_in = [], 3
        for i in range(4):
            c = width * 2 ** i
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True),
                       nn.Conv2d(c, c, 3, padding=1, bias=False), nn.BatchNorm2d(c), nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c
        self.backbone = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(dropout), nn.Linear(c_in, n_classes))
    def forward(self, x): return self.head(self.backbone(x))

class Pretrained(nn.Module):
    def __init__(self, n_classes, mode, dropout):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)      # weights come from the checkpoint
        self.backbone = nn.Sequential(*list(net.children())[:-1])
        self.head = nn.Sequential(nn.Flatten(), nn.Dropout(dropout), nn.Linear(net.fc.in_features, n_classes))
    def forward(self, x): return self.head(self.backbone(x))

def build(cfg):
    M = cfg['model']
    if M['name'] == 'SmallCNN': return SmallCNN(K, M['width'], M['dropout'])
    return Pretrained(K, M.get('mode'), M['dropout'])

MEAN = torch.tensor(ac.MEAN).view(1, 3, 1, 1); STD = torch.tensor(ac.STD).view(1, 3, 1, 1)
@torch.no_grad()
def logits_on_test(ck, m):                      # Step 8 code; the model is built and loaded beforehand
    m.eval()
    img, gpu = ck['img_size'], ck['config']['data'].get('gpu_aug', False)
    tf = None if gpu else ac.make_transforms(img, train=False)
    out = []
    for i in range(0, len(XT), 512):
        xb = XT[i:i + 512]
        if gpu:        # train_cls gpu_eval: bilinear + antialias resize, normalise
            x = torch.from_numpy(xb).to(DEVICE).permute(0, 3, 1, 2).float() / 255
            if x.shape[-1] != img: x = F.interpolate(x, size=(img, img), mode='bilinear', antialias=True, align_corners=False)
            x = (x - MEAN.to(DEVICE)) / STD.to(DEVICE)
        else:
            x = torch.stack([tf(np.array(a)) for a in xb]).to(DEVICE)
        with torch.autocast('cuda', enabled=DEVICE == 'cuda'): out.append(m(x).float().cpu())
    return torch.cat(out).numpy()


NETS = {}
for k, ck in CK.items():
    m = build(ck['config']); m.load_state_dict(ck['model']); NETS[k] = m.to(DEVICE).eval()   # fails here, not after the log
print(f'{len(NETS)} models built and loaded on {DEVICE}')
local = '/content/crops_test.npy'
if not os.path.exists(local) or os.path.getsize(local) != os.path.getsize(CROP_DIR / CROPS_USED['test']): shutil.copy(CROP_DIR / CROPS_USED['test'], local)
XT = np.load(local); assert len(XT) == len(TEST) == len(PRED), (len(XT), len(TEST), len(PRED))
print('test crops in memory:', XT.shape, '| nothing has been run on them yet')
for f in (PROBS_OUT, TOPK_OUT): assert not f.exists(), f

## 3. TestLog: opening 2 (last step before the first forward pass)

In [ ]:
entry = {c: '' for c in LOG.columns}
entry.update({'when': time.strftime('%Y-%m-%d %H:%M'), 'checkpoints': ';'.join(NAMES), 'reason': REASON})
if 'notebook_version' in LOG.columns: entry['notebook_version'] = 'step8b_topk'
NEW = pd.concat([LOG.assign(**({'reason': ''} if 'reason' not in LOG.columns else {})), pd.DataFrame([entry])], ignore_index=True)
NEW.to_csv(log_path, index=False)
print(f'TestLog now has {len(NEW)} entries:'); print(NEW.to_string(index=False))

## 4. Inference and the match check against TestPredictions_v1

In [ ]:
LOGITS, BAD, TIES = {}, [], []
for k, ck in CK.items():
    t0 = time.time(); L = logits_on_test(ck, NETS[k]); LOGITS[k] = L
    for kind, LL in (('raw', L), ('adjusted', L + np.array(ck['logit_bias']))):
        got = np.array(ac.CLASSES)[LL.argmax(1)]; want = PRED[f'{k[0].replace(" ", "")}_s{k[1]}_{kind}'].to_numpy()
        top2 = np.sort(LL, 1)[:, -2:]; gap = top2[:, 1] - top2[:, 0]
        diff = got != want; tie = diff & (gap < TIE_GAP); hard = diff & ~tie
        TIES.append({'model': k[0], 'seed': k[1], 'kind': kind, 'differ': int(diff.sum()), 'near_tie (allowed)': int(tie.sum()),
                     'other (stop)': int(hard.sum()), 'min gap of differing': float(gap[diff].min()) if diff.any() else None})
        if hard.any(): BAD.append((k, kind, int(hard.sum())))
    print(f'{k[0]} s{k[1]}: done ({time.time() - t0:.0f}s)')
T = pd.DataFrame(TIES); print(T.to_string(index=False))
assert not BAD, f'argmax differs from TestPredictions_v1 beyond a near-tie, nothing computed or saved: {BAD}'
print(f'match OK: {int(T["differ"].sum())} near-tie flips in total (gap < {TIE_GAP}), all others identical')

## 5. top-k accuracy and ECE, full test and without rgb

In [ ]:
def softmax(L): e = np.exp(L - L.max(1, keepdims=True)); return e / e.sum(1, keepdims=True)
def ece(P, Y, bins=15):                      # same as Step 8
    conf, pred = P.max(1), P.argmax(1); e = 0.0; edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any(): e += m.mean() * abs((pred[m] == Y[m]).mean() - conf[m].mean())
    return float(e)
def topk(P, Y, k): return float((np.argsort(-P, 1)[:, :k] == Y[:, None]).any(1).mean())

is_rgb = ((TEST.dataset == EXCLUDE[0]) & (TEST.source == EXCLUDE[1])).to_numpy()
CASES = {'full test': np.ones(len(TEST), bool), 'without rgb': ~is_rgb}
PROBS = {}
rows = []
for (m, s), L in LOGITS.items():
    for kind, LL in (('raw', L), ('adjusted', L + np.array(CK[(m, s)]['logit_bias']))):
        P = softmax(LL); PROBS[(m, s, kind)] = P
        for case, keep in CASES.items():
            rows.append({'model': m, 'seed': s, 'kind': kind, 'case': case, 'n_crops': int(keep.sum()),
                         **{f'top{k}_acc': topk(P[keep], YT[keep], k) for k in TOPK}, 'ece': ece(P[keep], YT[keep])})
R = pd.DataFrame(rows)
mets = [f'top{k}_acc' for k in TOPK] + ['ece']
agg = R.groupby(['case', 'model', 'kind'])[mets].agg(['mean', 'std'])
show = pd.DataFrame({c: [f'{a:.4f} ± {b:.4f}' for a, b in zip(agg[(c, 'mean')], agg[(c, 'std')])] for c in mets}, index=agg.index)
print(f'test, mean ± std over seeds {SEEDS} (rgb crops: {is_rgb.sum()}):'); print(show.to_string())

## 6. Save (new files only)

In [ ]:
out = TEST[['crop_id', 'dataset', 'category', 'source']].copy()
for (m, s, kind), P in PROBS.items():
    for i, c in enumerate(ac.CLASSES): out[f'{m.replace(" ", "")}_s{s}_{kind}_p_{c}'] = P[:, i].round(6)
for f in (PROBS_OUT, TOPK_OUT): assert not f.exists(), f
out.to_csv(PROBS_OUT, index=False); print('saved', PROBS_OUT.name, out.shape)
summ = agg.copy(); summ.columns = [f'{a}_{b}' for a, b in summ.columns]; summ = summ.reset_index().assign(row='mean_std')
pd.concat([R.assign(row='per_seed'), summ], ignore_index=True).to_csv(TOPK_OUT, index=False); print('saved', TOPK_OUT.name)